In [ ]:
import glob
import os

from natsort import natsorted
from pprint import pprint
from tqdm import tqdm

import numpy as np
import tifffile as tiff

from lib.visualization import plot_traces
from lib.visualization import show_one_image

In [ ]:
base_dir = "/home/wrx/Data_attention/Transfer learning/LinShu/DATA_linshu/"
data_type = "003 2P-V1"
mouse_id = "A455"
mouse_path = os.path.join(base_dir, data_type, mouse_id)
print(mouse_path)
os.path.exists(mouse_path)
session_path = natsorted(glob.glob(os.path.join(mouse_path, "2026*")))
pprint(session_path)

In [ ]:
# select which session to process
session_folder = session_path[6]
print(f"Session folder: {session_folder}")

In [ ]:
suite2p_folder = os.path.join(session_folder, "suite2p")
if not os.path.exists(suite2p_folder):
    print(f"Suite2p folder does not exist: {suite2p_folder}")
    os.makedirs(suite2p_folder)

raw_folder = glob.glob(os.path.join(session_folder, "TSeries-*"))
if not(len(raw_folder) == 1):
    print(f"Expected exactly one raw data folder, but found {len(raw_folder)}")
print(f"Raw data folder: {raw_folder[0]}")

In [ ]:
# get parameters of imaging data from the xml file in the raw data folder
from lib.input_output import get_parameters

imaging_parameters = get_parameters(raw_folder[0])

plane_num = imaging_parameters['plane_num']
print(f'detected plane_num: {plane_num}')

plane_ls = list(range(plane_num))
print(f'set plane_ls: {plane_ls}')

channel_ls = imaging_parameters['channel_ls'] # recorded channels
print(f'detected channel_ls: {channel_ls}')

frame_size = (int(imaging_parameters['pixelsPerLine']),
    int(imaging_parameters['linesPerFrame']))
print(f'detected frame_size: {frame_size}')

frame_rate = imaging_parameters['frame_rate']
print(f'detected frame_rate: {frame_rate}')

# Run the suite2p pipeline

Make ops

In [ ]:
import suite2p
from suite2p.io import ome_to_binary
ops = suite2p.default_ops()

ops['anatomical_only'] = 0
ops['baseline'] = 'maximin'
ops['bidiphase'] = 0
ops['bruker'] = True
ops['combined'] = False
ops['data_path'] = raw_folder

ops['do_bidiphase'] = True
ops['fs'] = frame_rate
ops['functional_chan'] = 2
ops['high_pass'] = 100
ops['input_format'] = 'bruker'
ops['max_overlap'] = 1.0
ops['nbinned'] = 5000

ops['preclassify'] = 0
ops['reg_tif'] = False
ops['reg_tif_chan2'] = False
ops['save_path0'] = session_folder
ops['sig_baseline'] = 30

ops['tau'] = 1.3

ops['win_baseline'] = 180

# Main settings
ops['multiplane_parallel'] = True
ops['nchannels'] = len(channel_ls)
ops['nplanes'] = len(plane_ls)

# File input/output settings
# ops['save_folder'] = suite2p_folder # seems not work for `ome_to_binary`

# Registration settings
# ops['align_by_chan'] = 2 # as the anatomical channel saved in the second channel
ops['do_registration'] = True
ops['keep_movie_raw'] = False # Whether to keep binary file of non-registered frames.
ops['maxregshift'] = 0.1 # in fraction of frame size
ops['nimg_init'] = 1000 # how many frames to use to compute reference image for registration
# ops['nonrigid'] = True
ops['two_step_registration'] = True # whether or not to run registration twice 

pprint(ops)

Registration

In [ ]:
# # Get the frame number of each plane for each stimulus.

# folder_name = os.path.basename(raw_folder[0])
# file_filter = '{}_Cycle00001_Ch2_*.ome.tif'.format(folder_name)
# n_frames = len(glob.glob('{}/{}'.format(raw_folder[0], file_filter)))
# print(f'n_frames: {n_frames}')
# # save the frame number of each plane for each stimulus to ops
# ops['n_frames'] = n_frames
# np.save(os.path.join(suite2p_folder, 'ops.npy'), ops)

In [ ]:
plane_folder = os.path.join(suite2p_folder, 'plane0')
f_data_path = os.path.join(plane_folder, 'data.bin')
if os.path.exists(f_data_path):
    print(f"Data already converted to binary format at: {f_data_path}")
else:
    print(f"Converting data to binary format at: {f_data_path}")
    ops0 = ome_to_binary(ops=ops) # convert all the planes together

In [ ]:
f_data = suite2p.io.BinaryFile(Ly=frame_size[1], Lx=frame_size[0],
    filename=f_data_path)
registration_outputs = suite2p.registration_wrapper(f_data, ops=ops)

In [ ]:
(refImg, rmin, rmax, meanImg, rigid_offsets, nonrigid_offsets, zest,
meanImg_chan2, badframes, yrange, xrange) = registration_outputs

np.save(os.path.join(plane_folder, 'rigid_offsets.npy'),
    np.array(rigid_offsets))

yoff, xoff, corrXY = rigid_offsets
offset = np.vstack((xoff, yoff)).T
plot_traces(offset, data_rate=frame_rate,
    labels=['xoff', 'yoff'], figsize=(10, 3), ylabel='offset (pixels)',
    title='Rigid offsets')

show_one_image(meanImg, figsize=(10, 10), title = "registered_mean_image")
tiff.imwrite(os.path.join(plane_folder, 'meanImg.tif'),
    meanImg.astype(np.int16))

In [ ]:
# Calculate the Mean and STD projection images across frames and save
f_reg_path = os.path.join(plane_folder, 'data.bin')
f_reg = suite2p.io.BinaryFile(Ly=frame_size[1], Lx=frame_size[0],
    filename=f_reg_path)
print('shape of f_reg:', f_reg.shape)

# from <class 'suite2p.io.binary.BinaryFile'> to <class 'numpy.memmap'>
f_reg = f_reg[:]

# to save disk space, not save the registered movie as tiff file
# movie_path = os.path.join(plane_folder, 'reg.tif')
# print('Writing to tiff file: {}'.format(movie_path))
# tiff.imwrite(movie_path, movie_list[stim])

mean_projection = np.mean(f_reg, axis=0)
tiff.imwrite(os.path.join(plane_folder, 'mean_projection.tif'),
    mean_projection.astype(np.int16))
std_projection = np.std(f_reg, axis=0)
tiff.imwrite(os.path.join(plane_folder, 'std_projection.tif'),
    std_projection.astype(np.int16))